# Cvičenie: Predikcia platu pomocou neurónovej siete

**Dataset:** `data/job_salary_prediction_dataset.csv`  
**Úloha:** Regresia — predikuj ročný plat na základe pracovných charakteristík  
**Architektúra:** MLP (Multi-Layer Perceptron)  

---

### Postup (workflow)
1. Načítaj a preskúmaj dáta
2. One-Hot Encoding kategorických stĺpcov
3. Rozdeľ dataset na train / val / test
4. Štandardizuj features (fit iba na train!)
5. Vytvor PyTorch DataLoader-y
6. Definuj model
7. Trénuj s early stopping
8. Vyhodnoť na testovacej množine

## 0. Importy

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Zariadenie: {device}")
kagglehub.dataset_download("nalisha/job-salary-prediction-dataset", output_dir="./data")


Zariadenie: cpu


'./data'

## 1. Načítanie a prieskum datasetu

Stĺpce v datasete:
- **Numerické:** `experience_years`, `skills_count`, `certifications`
- **Kategorické:** `job_title`, `education_level`, `industry`, `company_size`, `location`, `remote_work`
- **Target:** `salary` (ročný plat v USD)

In [2]:
df = pd.read_csv("data/job_salary_prediction_dataset.csv")

print(f"Tvar datasetu: {df.shape}")
print()
df.head()

Tvar datasetu: (250000, 10)



,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   job_title         250000 non-null  str  
 1   experience_years  250000 non-null  int64
 2   education_level   250000 non-null  str  
 3   skills_count      250000 non-null  int64
 4   industry          250000 non-null  str  
 5   company_size      250000 non-null  str  
 6   location          250000 non-null  str  
 7   remote_work       250000 non-null  str  
 8   certifications    250000 non-null  int64
 9   salary            250000 non-null  int64
dtypes: int64(4), str(6)
memory usage: 19.1 MB


In [4]:
df.describe()

,experience_years,skills_count,certifications,salary
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,10.005408,9.997812,2.491928,145718.080524
std,6.060602,5.479288,1.706475,37407.952729
min,0.000000,1.000000,0.000000,31867.000000
25%,5.000000,5.000000,1.000000,119358.000000
50%,10.000000,10.000000,2.000000,143453.000000
75%,15.000000,15.000000,4.000000,169492.000000
max,20.000000,19.000000,5.000000,333046.000000


In [ ]:
print("Chýbajúce hodnoty:")
print(df.isnull().sum())

In [ ]:
cat_cols = ["job_title", "education_level", "industry", "company_size", "location", "remote_work"]

# TODO: Pre každý stĺpec v cat_cols vypíš počet unikátnych hodnôt a ich zoznam
# Nápoveda: df[col].nunique() a df[col].unique()
for col in cat_cols:
    pass  # TODO: nahraď týmto správnym kódom

## 2. One-Hot Encoding (OHE)

Neurónová sieť pracuje iba s číslami. Kategorické stĺpce musíme zakódovať.

**One-Hot Encoding:** každá kategória dostane vlastný binárny stĺpec (0 alebo 1).  
Príklad pre `remote_work`:
```
remote_work    →   remote_work_Hybrid   remote_work_Yes
"No"                      0                   0
"Hybrid"                  1                   0
"Yes"                     0                   1
```
`drop_first=True` odstráni prvú kategóriu (predchádza multikolinearite).

**Dôležité:** OHE rob na celom DataFrame *pred* rozdelením na train/test,  
aby všetky sety mali rovnaké stĺpce.

In [ ]:
num_cols = ["experience_years", "skills_count", "certifications"]
target_col = "salary"

# TODO: Aplikuj pd.get_dummies na df, zakóduj stĺpce cat_cols, drop_first=True
# Výsledok ulož do df_encoded
# Nápoveda: pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_encoded = ???  # TODO

print(f"Počet stĺpcov pred OHE: {df.shape[1]}")
print(f"Počet stĺpcov po OHE:   {df_encoded.shape[1]}")
print()
print("Stĺpce po OHE:")
print(df_encoded.columns.tolist())

In [ ]:
# TODO: Vytvor X (numpy array všetkých stĺpcov okrem target_col) ako float32
# a y (numpy array stĺpca target_col) ako float32
# Nápoveda: df_encoded.drop(columns=[target_col]).values.astype("float32")
X = ???  # TODO
y = ???  # TODO

print(f"X tvar: {X.shape}")
print(f"y tvar: {y.shape}")
print(f"Rozsah platov: {y.min():.0f} – {y.max():.0f} USD")

## 3. Rozdelenie datasetu

Rozdeľujeme na **train / val / test** v pomere 70 % / 15 % / 15 %.

- **Train:** na trénovanie modelu (model vidí tieto dáta)
- **Val:** na sledovanie výkonu počas trénovania a early stopping
- **Test:** finálne vyhodnotenie (model ich *nikdy* nevidel počas tréningu)

Postup v dvoch krokoch:
1. Oddeľ test (15%) → dostaneš train+val (85%)
2. Z train+val oddeľ val (15/85 ≈ 17.6%) → dostaneš train (70%)

In [ ]:
# TODO: Rozdeľ X a y na (X_temp, X_test) a (y_temp, y_test)
# test_size=0.15, random_state=42
X_temp, X_test, y_temp, y_test = ???  # TODO

# TODO: Rozdeľ X_temp a y_temp na (X_train, X_val) a (y_train, y_val)
# test_size musí byť taký, aby val tvorilo 15% z pôvodného datasetu
# Nápoveda: 0.15 / 0.85 ≈ 0.176
X_train, X_val, y_train, y_val = ???  # TODO

print(f"Train: {X_train.shape[0]} vzoriek")
print(f"Val:   {X_val.shape[0]} vzoriek")
print(f"Test:  {X_test.shape[0]} vzoriek")

## 4. Štandardizácia features

Neurónovým sieťam sa lepšie trénuje, keď sú vstupné hodnoty v podobnom rozsahu (blízko 0, rozptyl ~1).

$$x' = \frac{x - \mu}{\sigma}$$

**Kľúčové pravidlo:** `μ` a `σ` vypočítaj **iba z trénovacej množiny**,  
potom ich aplikuj aj na val aj na test.  
Dôvod: pri nasadení modelu nemáš prístup k budúcim dátam — val/test simulujú túto situáciu.

In [ ]:
# TODO: Vypočítaj mu a sigma z X_train (axis=0, keepdims=True)
# Pridaj 1e-6 k sigma, aby si predišiel deleniu nulou
mu    = ???  # TODO
sigma = ???  # TODO

# TODO: Štandardizuj X_train, X_val a X_test pomocou mu a sigma
X_train_s = ???  # TODO
X_val_s   = ???  # TODO
X_test_s  = ???  # TODO

print(f"X_train_s — priemer: {X_train_s.mean():.4f}, std: {X_train_s.std():.4f}")
print(f"X_val_s   — priemer: {X_val_s.mean():.4f}, std: {X_val_s.std():.4f}")

## 5. PyTorch DataLoader

PyTorch pracuje s `Tensor`-mi, nie s numpy poľami.  
`TensorDataset` zbalí X a y do jedného objektu, `DataLoader` z neho robí mini-batche.

Pre **train** nastavíme `shuffle=True` (náhodné poradie batchov — zabraňuje memorovaniu poradia).  
Pre **val/test** `shuffle=False` (poradie nevadí, len vyhodnocujeme).

In [ ]:
BATCH_SIZE = 64

# TODO: Preveď X_train_s, X_val_s, X_test_s na torch.FloatTensor
# a y_train, y_val, y_test na torch.FloatTensor s tvarom [-1, 1]
# Nápoveda: torch.FloatTensor(pole).unsqueeze(1) pre y (pridá dimenziu → [N,1])
X_train_t = ???  # TODO
y_train_t = ???  # TODO
X_val_t   = ???  # TODO
y_val_t   = ???  # TODO
X_test_t  = ???  # TODO
y_test_t  = ???  # TODO

# TODO: Vytvor TensorDataset pre train, val a test
train_ds = ???  # TODO
val_ds   = ???  # TODO
test_ds  = ???  # TODO

# TODO: Vytvor DataLoader pre train (shuffle=True), val a test (shuffle=False)
train_loader = ???  # TODO
val_loader   = ???  # TODO
test_loader  = ???  # TODO

# Overenie
X_batch, y_batch = next(iter(train_loader))
print(f"Batch X tvar: {X_batch.shape}")
print(f"Batch y tvar: {y_batch.shape}")

## 6. Definícia modelu

Pre **regresiu** (predikcia spojitej hodnoty) platia tieto pravidlá:
- Výstupná vrstva má **1 neurón**
- **Žiadna aktivácia** na výstupe (sieť môže predpovedať ľubovoľné číslo)
- Loss funkcia: `MSELoss` (Mean Squared Error)

Architektúra:
```
vstup (input_dim) → Linear → ReLU → Linear → ReLU → Linear → výstup (1)
                    [128]           [64]              
```

In [ ]:
# TODO: Definuj triedu SalaryMLP(nn.Module)
# __init__: prijme input_dim, vytvor nn.Sequential so štruktúrou:
#   Linear(input_dim → 128) → ReLU → Linear(128 → 64) → ReLU → Linear(64 → 1)
# forward: aplikuj self.net na vstup x
class SalaryMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = ???  # TODO

    def forward(self, x):
        return ???  # TODO


input_dim = X_train_t.shape[1]
model = SalaryMLP(input_dim).to(device)
print(model)
print(f"\nPočet parametrov: {sum(p.numel() for p in model.parameters()):,}")

## 7. Konfigurácia trénovania

- **Loss:** `MSELoss` — minimalizuje priemerný kvadratický rozdiel medzi predikciou a skutočnosťou
- **Optimizer:** `Adam` s learning rate 0.001 — adaptívny, vhodný pre väčšinu sietí
- **Early stopping:** ak sa val loss nezlepší za `patience` epôch, zastavíme tréning

In [ ]:
criterion = nn.MSELoss()

# TODO: Vytvor Adam optimizer pre parametre modelu, learning_rate=1e-3
# Nápoveda: torch.optim.Adam(model.parameters(), lr=...)
opt = ???  # TODO

EPOCHS  = 200
PATIENCE = 20

## 8. Trénovanie

### Funkcia `run_epoch`

Táto funkcia prechádza celý loader (všetky mini-batche) a vracia priemernú loss a MAE.

- `training=True` → model sa trénuje (aktualizujeme váhy)
- `training=False` → model vyhodnocujeme (váhy sa nemenia)

**MAE** (Mean Absolute Error) = priemerná absolútna chyba v USD — ľahšie interpretovateľná ako MSE.

In [ ]:
def run_epoch(loader, training: bool):
    """Prechod cez loader. Vracia (avg_mse_loss, avg_mae)."""
    # TODO: Prepni model do správneho módu: model.train(training)
    ???  # TODO

    total_loss = 0.0
    total_mae  = 0.0
    total_n    = 0

    # TODO: Zvol správny kontext — torch.enable_grad() pre tréning, torch.no_grad() pre eval
    ctx = ???  # TODO
    with ctx:
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)

            if training:
                # TODO: Vynuluj gradienty
                ???  # TODO

            # TODO: Predikcia modelu
            preds = ???  # TODO

            # TODO: Vypočítaj MSE loss medzi preds a y_b
            loss = ???  # TODO

            if training:
                # TODO: Spätné šírenie chyby a krok optimizéra
                ???  # TODO
                ???  # TODO

            n = X_b.size(0)
            total_loss += loss.item() * n
            # TODO: Pripočítaj MAE k total_mae (použij torch.abs a .mean())
            total_mae  += ???  # TODO
            total_n    += n

    return total_loss / total_n, total_mae / total_n

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}

best_val_loss = float("inf")
best_state    = None
patience_cnt  = 0

for epoch in range(1, EPOCHS + 1):
    # TODO: Spusti run_epoch pre train loader (training=True)
    train_loss, train_mae = ???  # TODO

    # TODO: Spusti run_epoch pre val loader (training=False)
    val_loss, val_mae = ???  # TODO

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_mae"].append(train_mae)
    history["val_mae"].append(val_mae)

    if epoch % 10 == 0:
        print(f"Ep {epoch:3d} | train MSE: {train_loss:10.1f}  MAE: {train_mae:7.1f} "
              f"| val MSE: {val_loss:10.1f}  MAE: {val_mae:7.1f}")

    # TODO: Implementuj early stopping
    # Ak val_loss < best_val_loss - 1e-4:
    #   aktualizuj best_val_loss, ulož stav modelu do best_state, resetuj patience_cnt
    # Inak:
    #   zvýš patience_cnt; ak dosiahol PATIENCE → obnov best_state a break
    # Nápoveda: {k: v.clone() for k, v in model.state_dict().items()}
    ???  # TODO

print("\nTréning dokončený.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train MSE")
axes[0].plot(history["val_loss"],   label="Val MSE")
axes[0].set_title("MSE Loss")
axes[0].set_xlabel("Epocha")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history["train_mae"], label="Train MAE")
axes[1].plot(history["val_mae"],   label="Val MAE")
axes[1].set_title("MAE (USD)")
axes[1].set_xlabel("Epocha")
axes[1].set_ylabel("MAE [USD]")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 9. Vyhodnotenie na testovacej množine

Test set model **nikdy nevidel** počas trénovania — dáva reálny odhad výkonu na nových dátach.

Vizualizácie:
- **Scatter plot** predikcia vs. skutočnosť — body blízko diagonály = dobrý model
- **Histogram reziduálov** — rozdelenie chýb by malo byť symetrické okolo 0

In [ ]:
# TODO: Spusti run_epoch pre test loader (training=False) a vypíš test MAE
test_loss, test_mae = ???  # TODO
print(f"Test MSE: {test_loss:,.1f}")
print(f"Test MAE: {test_mae:,.1f} USD")

In [ ]:
# TODO: Zbieraj predikcie pre celý test set
# Prejdi cez test_loader, pre každý batch zavolaj model a zbieraj predikcie + skutočné hodnoty
# Nápoveda: preds_list.append(model(X_b).cpu().detach().numpy())
model.eval()
all_preds  = []
all_actual = []

with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        # TODO: Predikuj a pripoj výsledky do all_preds a all_actual
        ???  # TODO

all_preds  = np.concatenate(all_preds).flatten()
all_actual = np.concatenate(all_actual).flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot
axes[0].scatter(all_actual, all_preds, alpha=0.4, s=15)
mn = min(all_actual.min(), all_preds.min())
mx = max(all_actual.max(), all_preds.max())
axes[0].plot([mn, mx], [mn, mx], "r--", label="Ideálna predikcia")
axes[0].set_xlabel("Skutočný plat [USD]")
axes[0].set_ylabel("Predikovaný plat [USD]")
axes[0].set_title("Predikcia vs. Skutočnosť")
axes[0].legend()
axes[0].grid(True)

# Histogram reziduálov
residuals = all_preds - all_actual
axes[1].hist(residuals, bins=40, edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", label="Nulová chyba")
axes[1].set_xlabel("Reziduál (predikcia − skutočnosť) [USD]")
axes[1].set_ylabel("Počet vzoriek")
axes[1].set_title("Distribúcia chýb")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Bonusové úlohy (voliteľné)

1. **Dropout:** Pridaj `nn.Dropout(0.3)` po každej ReLU vrstve. Pomôže to? Zmeň sa val MAE?
2. **Hlbšia sieť:** Vyskúšaj architektúru `input → 256 → 128 → 64 → 1`. Čo sa zmení?
3. **Iný optimizer:** Nahraď Adam za SGD s learning rate 0.01. Porovnaj rýchlosť konvergencie.
4. **Normalizácia targetu:** Štandardizuj aj `y` (plat). Pomôže to trénovaniu?
5. **Feature importance:** Nastav všetky hodnoty jedného feature na 0 a sleduj zmenu MAE — ktorý feature je najdôležitejší?